# Carga del archivo e Importación de Librerías

In [1]:

import numpy as np 
import pandas as pd 
import scipy.stats as stats
from statsmodels.stats.proportion import proportions_ztest # Prueba de z de proporcines 
from statsmodels.stats.multitest import multipletests #para corrección bonferroni

df = pd.read_excel("diabetes.xlsx")
pd.set_option("display.max_rows", None)  # Mostrar todas las filas
pd.set_option("display.max_columns", None)  # Mostrar todas las columnas

# 1

¿El  valor  promedio  de  colesterol  total  de  la  muestra,  correspondiente  a 
pacientes masculinos es menor 200mg/dL Y en las pacientes femeninas, es 
menor 200mg/dL?

- Variable: s1 $\rightarrow$ total serum cholesterol
- Hipótesis:
    - $H_0$: El valor  promedio  de  colesterol  total  de  la  muestra es igual o mayor a 200 mg/dL.
    - $H_1$: El valor  promedio  de  colesterol  total  de  la  muestra es menor a 200mg/dL para pacientes másculinos y femeninos.

Para probar la normalidad de los datos, hacemos una prueba de normalidad.

In [2]:
numero_masc = len(df[df['SEX'] == 2]['S1'])

print(f"Total Masculino {numero_masc}")

numero_fem = len(df[df['SEX'] == 1]['S1'])

print(f"Total Femenino {numero_fem}")

Total Masculino 207
Total Femenino 235


Como el total de datos es mayor a 50 en ambos casos, usamos la prueba de Kolmogorov-Smirnoff.

- Definición de Hipótesis para Kolmogorov-Smirnoff
    - H0(Hipótesis Nula): Los datos sí siguen una distribución normal.
    - H1(Hipótesis Alterna): Los datos no siguen una distribución normal.

In [ ]:
# 1. Filtramos los datos: Solo columna 'S1' donde 'SEX' sea 2 (Hombres)
datos_hombres_s1 = df[df['SEX'] == 2]['S1']

# 2. Calculamos media y desviación estándar de ESOS datos específicamente
mu = np.mean(datos_hombres_s1)
sigma = np.std(datos_hombres_s1)

# 3. Realizamos la prueba Kolmogorov-Smirnov solo para ese grupo
ks_stat, ks_p = stats.kstest(datos_hombres_s1, 'norm', args=(mu, sigma))


# 4. Decisión
alpha = 0.05
print("Para hombres:")
if ks_p > alpha:
    print(f"Interpretación: El p-valor ({ks_p}) es mayor a 0.05. Asumimos que los datos de S1 en hombres son NORMALES.")
else:
    print(f"Interpretación: El p-valor ({ks_p}) es menor a 0.05. Asumimos que los datos son ANORMALES.")
#------------------------------------------------------------

# 1. Filtramos los datos: Solo columna 'S1' donde 'SEX' sea 2 (Mujeres)
datos_mujeres_s1 = df[df['SEX'] == 1]['S1']

# 2. Calculamos media y desviación estándar de ESOS datos específicamente
mu = np.mean(datos_mujeres_s1)
sigma = np.std(datos_mujeres_s1)

# 3. Realizamos la prueba Kolmogorov-Smirnov solo para ese grupo
ks_stat, ks_p = stats.kstest(datos_mujeres_s1, 'norm', args=(mu, sigma))


# 4. Decisión
alpha = 0.05
print("Para mujeres:")
if ks_p > alpha:
    print(f"Interpretación: El p-valor ({ks_p}) es mayor a 0.05. Asumimos que los datos de S1 en mujeres son NORMALES.")
else:
    print(f"Interpretación: El p-valor ({ks_p}) es menor a 0.05. Asumimos que los datos son ANORMALES.")

Para hombres:
Interpretación: El p-valor (0.2396770979328262) es mayor a 0.05. Asumimos que los datos de S1 en hombres son NORMALES.
Para mujeres:
Interpretación: El p-valor (0.7024136704056544) es mayor a 0.05. Asumimos que los datos de S1 en mujeres son NORMALES.


Como en ambos casos, la distribución es normal, podemos efectuar la prueba t

In [22]:
# Para hombres
data = df[df['SEX'] == 2]['S1'] # Datos de colesterol total para hombres
mu = 200 # El valor de referencia contra el cual comparamos

# Ejecución
t_stat, V_pMasc = stats.ttest_1samp(data, mu, alternative='less')

# Para Mujeres
data = df[df['SEX'] == 1]['S1'] # Datos de colesterol total para hombres
mu = 200 # El valor de referencia contra el cual comparamos

# Ejecución
t_stat, V_pFem = stats.ttest_1samp(data, mu, alternative='less')

alpha = 0.05

if V_pMasc < alpha:
    print("Para Hombres H0 es Falso")
else:
    print("Para Hombres H0 es Verdadero")

if V_pFem < alpha:
    print("Para Mujeres H0 es Falso")
else:
    print("Para Mujeres H0 es Verdadero")


Para Hombres H0 es Falso
Para Mujeres H0 es Falso


Por lo tanto, tanto para hombres y mujeres, $H_1$ es verdadero; lo que significa que el valor  promedio  de  colesterol  total  de  la  muestra es menor a 200mg/dL para pacientes másculinos y femeninos.

# 2

¿Hay diferencias entre el nivel de colesterol ldl entre pacientes con menos 
de 40 años y más de 40 años de edad?

- Grupos: 
    - Grupo 1: Pacientes con menos de 40 años (n=117).
    - Grupo 2: Pacientes con más de 40 años (n=320).

In [24]:

# Contar pacientes con edad mayor a 40
pacientes_mayores_40 = len(df[df['AGE'] > 40])

print(f"Total de pacientes con edad mayor a 40: {pacientes_mayores_40}")


# Contar pacientes con edad menor a 40
pacientes_menores_40 = len(df[df['AGE'] < 40])

print(f"Total de pacientes con edad menor a 40: {pacientes_menores_40}")

Total de pacientes con edad mayor a 40: 320
Total de pacientes con edad menor a 40: 117
